# Chapter 5 - Lab 2: The Full Deep Search Pipeline (customer-care edition)

**Goal**: wire four components into one autonomous research pipeline:

1. **Planner** (Lab 1) decomposes the question into a dependency graph of sub-tasks.
2. **Researcher** executes the graph in parallel "waves" with `asyncio.gather`.
3. **Validator** checks the evidence with **deterministic Python**, not an LLM.
4. **Synthesizer** writes a structured CX research memo (`CXResearchReport`).

If validation fails, the gaps are fed back to the planner (**replanning**).

*Finance edition*: NVIDIA vs AMD vs Intel investment memo. *Customer-care edition*: root causes of a
CSAT drop for the FrostLine refrigerator, compared with two healthy product lines.

**Tech stack**: PydanticAI (planner and synthesizer with structured output), OpenAI `gpt-4o`, Python `asyncio`. Data tools are stubs so the lab runs with only an OpenAI key.

## 1. Install packages

In [ ]:
%pip install -q 'pydantic-ai-slim[openai]' pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared models and helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%205/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (ResearchPlan, SubTask, TaskStatus, ProductLineMetrics, ValidationResult,
                    CXResearchReport, PLANNING_MODEL, SYNTHESIS_MODEL, MAX_REPLAN_ATTEMPTS,
                    extract_product_line)
print("Shared models loaded.")

## 4. Define stub tools

Each stub is **async** with the same signature as the real tool in `src/tools/`, so swapping in real data sources later changes nothing else.

In [ ]:
import asyncio
from common import ProductLineMetrics

# Stub data (Q3 2026). In production: the support warehouse, the document store, and a web-search API.
_METRICS = {
    "FL-340":  dict(product_name="FrostLine Refrigerator 340L", orders=3150, tickets=1134, contact_rate=36.0,
                    csat=3.4, csat_change_qoq=-0.7, first_contact_resolution=48.0, repeat_contact_rate=38.0,
                    avg_resolution_hours=71.0, return_rate=4.2),
    "ZX1":     dict(product_name="Zenith X1 Smartphone", orders=18400, tickets=2760, contact_rate=15.0,
                    csat=4.2, csat_change_qoq=-0.1, first_contact_resolution=74.0, repeat_contact_rate=14.0,
                    avg_resolution_hours=19.0, return_rate=3.1),
    "AQP-500": dict(product_name="AquaPure RO Purifier", orders=5200, tickets=832, contact_rate=16.0,
                    csat=4.4, csat_change_qoq=0.1, first_contact_resolution=81.0, repeat_contact_rate=9.0,
                    avg_resolution_hours=12.0, return_rate=1.8),
}
_DOCS = {
    "FL-340": ("[KIB-2026-014] Compressor relay R-22: 3.1% early failures in July-August 2026 builds; "
               "spare back-ordered until 15 Nov. [Field service] First-visit fix rate 52% vs 85% norm. "
               "[QA audit] 40% of escalated calls say 'explain again'; replacement under KB-102 rarely "
               "offered before the third contact."),
    "ZX1": "[Logistics] 61% of ZX1 tickets are 'where is my order' contacts caused by the Delhi hub backlog.",
    "AQP-500": "[CX initiative] Filter how-to video in August cut how-to tickets by 35%.",
}
_WEB = ("[Twitter, SYNTHETIC] '#NovaMartFail 3 weeks without a working fridge' (1.2k reposts). "
        "[Forum, SYNTHETIC] 'FrostLine 340L stops cooling after a month', 46 replies. "
        "[Competitor, SYNTHETIC] CoolHome launches a 7-day replacement guarantee.")

async def get_product_line_metrics(code: str) -> ProductLineMetrics:
    await asyncio.sleep(0.2)   # simulated latency
    return ProductLineMetrics(product_line=code, period="Q3 2026", **_METRICS[code])

async def get_cx_documents(code: str) -> str:
    await asyncio.sleep(0.2)
    return f"Internal CX documents for {code}: {_DOCS.get(code, 'none')}"

async def search_cx_web(query: str) -> str:
    await asyncio.sleep(0.2)
    return _WEB

## 5. The planner (from Lab 1)

In [ ]:
PLANNER_SYSTEM_PROMPT = """\
You are a customer-experience research planner at a retailer. Given a research
question, decompose it into concrete sub-tasks that can be executed by tools.

Available data sources for each sub-task:
- support_analytics: Structured service metrics per product line (orders, tickets,
  contact rate, CSAT and its change, first-contact resolution, repeat-contact rate,
  resolution time, return rate). Best for quantitative data.
- cx_documents: Internal documents per product line (engineering known-issue bulletins,
  field-service reports, QA call audits). Best for root causes.
- web_search: Social media, review sites, consumer forums, competitor moves.
  Best for external sentiment and market context.

Rules for creating the plan:
1. Each sub-task must have a clear, measurable outcome (e.g., "Fetch FL-340 CSAT,
   contact rate and FCR for Q3" not "Understand the fridge problem").
2. Name the product-line code (e.g. FL-340, ZX1, AQP-500) in every product-specific task.
3. Specify which data_sources each sub-task should use (can be multiple).
4. Mark dependencies: which task IDs must complete before this task can start.
5. Tasks with no dependencies can execute in parallel: maximise parallelism.
6. For multi-product comparisons, create parallel tasks for each product line.
7. Include a comparative analysis task that depends on all product data tasks.
8. The final task should synthesise all findings: it depends on everything.
9. Use sequential IDs starting from 1.
10. Aim for 5-10 sub-tasks. Too few = insufficient depth. Too many = overhead.
11. If the question is about a single product line, still gather a comparison line.\
"""

from pydantic_ai import Agent

planner_agent = Agent(model=PLANNING_MODEL, system_prompt=PLANNER_SYSTEM_PROMPT,
                      output_type=ResearchPlan, retries=2)

async def create_research_plan(question: str, additional_context: str = "") -> ResearchPlan:
    msg = question + (f"\n\nAdditional context:\n{additional_context}" if additional_context else "")
    return (await planner_agent.run(msg)).output

## 6. The researcher: parallel DAG execution

Each iteration collects the *ready* tasks (all dependencies completed) and runs them concurrently.
This is topological "wave-front" execution: the metrics for three product lines, the internal documents and
the social-media search all run at the same time; the comparison waits for them.

In [ ]:
analyst_agent = Agent(
    model=SYNTHESIS_MODEL,
    system_prompt=("You are a senior customer-experience analyst. Use specific numbers from the data. "
                   "Do not fabricate data. Separate symptoms (low CSAT) from root causes (a missing part)."),
    retries=1,
)

async def _llm_analyze(description: str, context: str) -> str:
    result = await analyst_agent.run(f"Task: {description}\n\nAvailable data:\n{context or 'No prior data.'}")
    return result.output

In [ ]:
async def execute_plan(plan: ResearchPlan) -> dict[int, str]:
    """Execute a research plan, respecting task dependencies.

    Tasks with no unmet dependencies execute in parallel. Each wave of
    parallel tasks completes before the next wave of now-unblocked tasks begins.
    """
    results: dict[int, str] = {}
    tasks_by_id = {t.id: t for t in plan.sub_tasks}

    while any(t.status == TaskStatus.PENDING for t in plan.sub_tasks):
        ready = [
            t for t in plan.sub_tasks
            if t.status == TaskStatus.PENDING
            and all(tasks_by_id[dep].status == TaskStatus.COMPLETED for dep in t.dependencies)
        ]
        if not ready:
            for t in plan.sub_tasks:
                if t.status == TaskStatus.PENDING:
                    t.status = TaskStatus.FAILED
                    t.result = "Could not execute: unresolved dependencies"
                    results[t.id] = t.result
            break

        async def _execute_one(task: SubTask) -> tuple[int, str]:
            task.status = TaskStatus.IN_PROGRESS
            try:
                result = await _route_and_execute(task, results)
                if not isinstance(result, str) or not result.strip():
                    raise ValueError("Task returned no evidence")
                task.status = TaskStatus.COMPLETED
                task.result = result
                return task.id, result
            except Exception as e:
                task.status = TaskStatus.FAILED
                task.result = f"[FAILED] Task {task.id}: {e}"
                return task.id, task.result

        completed = await asyncio.gather(*[_execute_one(t) for t in ready])
        for task_id, result in completed:
            results[task_id] = result
            print(f"    [{tasks_by_id[task_id].status.value}] Task {task_id}: "
                  f"{tasks_by_id[task_id].description[:60]}")
    return results

In [ ]:
async def _route_and_execute(task: SubTask, prior_results: dict[int, str]) -> str:
    """Route a sub-task to the appropriate tool(s) and execute it."""
    context = ""
    if task.dependencies:
        context = "\n\n---\n\n".join(prior_results[d] for d in task.dependencies if d in prior_results)

    sources = task.data_sources

    # Support analytics: structured metrics
    if "support_analytics" in sources and not task.dependencies:
        code = extract_product_line(task.description)
        if code:
            metrics = await get_product_line_metrics(code)
            return metrics.model_dump_json(indent=2)

    # Internal documents: qualitative root-cause evidence
    if "cx_documents" in sources and "support_analytics" not in sources:
        code = extract_product_line(task.description)
        if code:
            return await get_cx_documents(code)

    # Web / social search
    if "web_search" in sources and len(sources) == 1:
        return await search_cx_web(task.description)

    # Mixed sources or analysis tasks: reason over prior findings with the LLM
    return await _llm_analyze(task.description, context)


async def _llm_analyze(description: str, context: str) -> str:
    """Use the analyst agent for tasks that need reasoning rather than a tool call."""
    prompt = f"Task: {description}\n\nAvailable data:\n{context if context else 'No prior data.'}"
    result = await analyst_agent.run(prompt)
    return result.output

## 7. The validator (deterministic checks)

Validation is **deterministic Python**, not an LLM call: faster, cheaper and far more reliable for questions
like "is every product line covered?" and "is this number even possible?". Two customer-care consistency checks:

* the **contact rate** must equal tickets / orders x 100;
* **first-contact resolution + repeat-contact rate** cannot exceed 100% (a ticket cannot be both).

In [ ]:
import math

def validate_research(plan: ResearchPlan, results: dict[int, str],
                      required_product_lines: list[str]) -> ValidationResult:
    """Validate research findings using deterministic checks."""
    errors: list[str] = []
    warnings: list[str] = []
    gaps: list[str] = []

    # Check 1: task completion
    if not plan.sub_tasks:
        gaps.append("Research plan contains no tasks")
    for task in plan.sub_tasks:
        if task.status != TaskStatus.COMPLETED:
            gaps.append(f"Task {task.id} is not completed: {task.description}")
    completed = sum(1 for t in plan.sub_tasks if t.status == TaskStatus.COMPLETED)
    total = len(plan.sub_tasks)
    if total and completed < total:
        pct = completed / total * 100
        if pct < 50:
            errors.append(f"Only {completed}/{total} tasks completed ({pct:.0f}%) - insufficient data for synthesis")
        else:
            warnings.append(f"{completed}/{total} tasks completed ({pct:.0f}%)")

    # Check 2: only completed tasks with parsed metrics establish coverage
    found: set[str] = set()
    parsed: list[ProductLineMetrics] = []
    for task in plan.sub_tasks:
        if task.status != TaskStatus.COMPLETED:
            continue
        result = results.get(task.id)
        if not isinstance(result, str) or not result.strip():
            gaps.append(f"Task {task.id} has no result")
            continue
        try:
            metrics = ProductLineMetrics.model_validate_json(result)
        except (ValueError, TypeError):
            if "support_analytics" in task.data_sources and not task.dependencies:
                errors.append(f"Task {task.id} did not return valid support metrics")
            continue  # qualitative text cannot establish metrics coverage
        parsed.append(metrics)
        _validate_metrics(metrics, errors, warnings)
        found.add(metrics.product_line.upper())

    for code in sorted({p.upper() for p in required_product_lines} - found):
        gaps.append(f"Missing structured support metrics for required product line: {code}")

    # Check 3: cross-product consistency
    if len(parsed) >= 2:
        periods = {m.period for m in parsed}
        if len(periods) > 1:
            detail = ", ".join(f"{m.product_line}={m.period}" for m in parsed)
            warnings.append(f"Period mismatch across product lines: {detail}. Comparisons may not be like-for-like.")

    return ValidationResult(is_valid=not errors and not gaps, errors=errors, warnings=warnings, gaps=gaps)


def _validate_metrics(m: ProductLineMetrics, errors: list[str], warnings: list[str]) -> None:
    """Validate one ProductLineMetrics instance for plausibility and internal consistency."""
    code = m.product_line
    if not m.period or not m.period.strip():
        errors.append(f"{code}: missing reported period")
        return
    for field in ("contact_rate", "csat", "first_contact_resolution", "repeat_contact_rate",
                  "avg_resolution_hours"):
        value = getattr(m, field)
        if value is None or not math.isfinite(value):
            errors.append(f"{code}: {field} must be a finite number")
            return

    if m.orders <= 0:
        errors.append(f"{code}: orders must be positive (got {m.orders})")
        return
    if not 1 <= m.csat <= 5:
        errors.append(f"{code}: CSAT {m.csat} outside the 1-5 scale")
    for field in ("first_contact_resolution", "repeat_contact_rate", "return_rate"):
        value = getattr(m, field)
        if value is not None and not 0 <= value <= 100:
            errors.append(f"{code}: {field} {value}% outside 0-100%")

    # Consistency 1: the contact rate must match tickets / orders.
    implied = m.tickets / m.orders * 100
    if abs(implied - m.contact_rate) > 0.5:
        errors.append(f"{code}: contact rate {m.contact_rate} does not match tickets/orders = {implied:.1f}")

    # Consistency 2: a ticket cannot be both solved at first contact and need a repeat contact.
    if m.first_contact_resolution + m.repeat_contact_rate > 101:
        errors.append(f"{code}: first-contact resolution ({m.first_contact_resolution}%) + repeat-contact "
                      f"rate ({m.repeat_contact_rate}%) exceed 100% - mathematically impossible")

    # Plausibility warnings (not errors)
    if m.contact_rate > 50:
        warnings.append(f"{code}: contact rate {m.contact_rate} per 100 orders is extreme - verify ticket mapping")
    if m.csat_change_qoq is not None and abs(m.csat_change_qoq) >= 0.5:
        warnings.append(f"{code}: CSAT moved {m.csat_change_qoq:+.1f} points in one quarter - a material shift")

## 8. The synthesizer (structured report)

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """\
You are the head of customer-experience analytics at a large retailer. Produce a
structured research memo for the Chief Customer Officer from the data gathered below.

Format requirements:
1. **Title**: clear and specific (e.g., "Why FrostLine CSAT fell in Q3 2026: a parts and process failure").
2. **Executive summary**: 3-5 sentences; lead with the most important insight.
3. **Product metrics**: include every product line analysed (structured separately).
4. **Root-cause analysis**: 2-3 paragraphs separating symptoms (CSAT, contact rate) from root
   causes (parts, process, policy, logistics). Use specific numbers and cite the documents.
5. **Risk factors**: ranked by severity and likelihood (churn, reputation, legal, competitor moves).
6. **Recommendations**: 3-5 concrete actions with an owner and an expected KPI effect.
7. **Conclusion**: direct answer to the research question.
8. **Sources**: every data source and document ID used.

Critical rules:
- Be PRECISE with numbers; use the exact figures from the gathered data.
- NEVER fabricate data. If a metric is missing, write "Data not available".
- Label synthetic web results as synthetic if they are marked so.
- If the data contains validation warnings or gaps, acknowledge them.
- Assign a confidence score (0-1) reflecting data completeness and quality.\
"""

synthesizer_agent = Agent(model=SYNTHESIS_MODEL, system_prompt=SYNTHESIZER_SYSTEM_PROMPT,
                          output_type=CXResearchReport, retries=2)

async def synthesize_report(question: str, results: dict[int, str],
                            validation: ValidationResult) -> CXResearchReport:
    """Synthesise validated findings into a structured research report."""
    findings = "\n\n".join(f"=== Task {tid} Result ===\n{results[tid]}" for tid in sorted(results))
    context = findings
    if validation.warnings:
        context += "\n\n=== Validation Warnings ===\n" + "\n".join(f"- {w}" for w in validation.warnings)
    if validation.gaps:
        context += "\n\n=== Data Gaps (acknowledged) ===\n" + "\n".join(f"- {g}" for g in validation.gaps)
    result = await synthesizer_agent.run(f"Research question: {question}\n\nGathered data and analysis:\n{context}")
    return result.output

## 9. The orchestrator: Plan -> Execute -> Validate -> Synthesize

In [ ]:
async def deep_search(question: str, product_lines: list[str]) -> CXResearchReport:
    additional_context = ""
    for attempt in range(MAX_REPLAN_ATTEMPTS + 1):
        print(f"[Plan] attempt {attempt + 1}")
        plan = await create_research_plan(question, additional_context)
        for t in plan.sub_tasks:
            print(f"   [{t.id}] {t.description[:70]}  deps={t.dependencies}")
        print("[Execute]")
        results = await execute_plan(plan)
        print("[Validate]")
        validation = validate_research(plan, results, product_lines)
        print("   valid:", validation.is_valid, "| warnings:", validation.warnings, "| gaps:", validation.gaps)
        if validation.is_valid:
            break
        additional_context = "Fix these issues:\n" + "\n".join(f"- {g}" for g in validation.gaps + validation.errors)
    else:
        raise RuntimeError("Validation failed after all replans; synthesis blocked.")
    print("[Synthesize]")
    return await synthesize_report(question, results, validation)

## 10. Run the full pipeline

In [ ]:
question = (
    "Why did customer satisfaction for the FrostLine refrigerator line (FL-340) fall in Q3 2026? "
    "Compare it with the Zenith X1 smartphone (ZX1) and the AquaPure purifier (AQP-500) on CSAT, "
    "contact rate, first-contact resolution and repeat-contact rate, identify root causes and risks."
)

report = await deep_search(question, ["FL-340", "ZX1", "AQP-500"])

## 11. Inspect the report

In [ ]:
def format_report_markdown(report: CXResearchReport) -> str:
    """Convert a CXResearchReport to a Markdown document."""
    def fmt(value, pattern):
        return pattern.format(value) if value is not None else "N/A"

    lines = [f"# {report.title}", "", f"**Confidence Score: {report.confidence_score:.0%}**", "", "---", "",
             "## Executive Summary", "", report.executive_summary, ""]
    if report.product_metrics:
        ms = report.product_metrics
        lines += ["## Key Metrics Comparison", "",
                  "| Metric | " + " | ".join(m.product_line for m in ms) + " |",
                  "| --- | " + " | ".join("---" for _ in ms) + " |"]
        rows = [("Orders", "orders", "{:,}"), ("Tickets", "tickets", "{:,}"),
                ("Contact rate (per 100 orders)", "contact_rate", "{:.1f}"),
                ("CSAT (1-5)", "csat", "{:.1f}"), ("CSAT change QoQ", "csat_change_qoq", "{:+.1f}"),
                ("First-contact resolution", "first_contact_resolution", "{:.0f}%"),
                ("Repeat-contact rate", "repeat_contact_rate", "{:.0f}%"),
                ("Avg resolution hours", "avg_resolution_hours", "{:.0f}"),
                ("Return rate", "return_rate", "{:.1f}%")]
        for label, attr, pattern in rows:
            lines.append(f"| {label} | " + " | ".join(fmt(getattr(m, attr), pattern) for m in ms) + " |")
        lines.append("")
    lines += ["## Root-Cause Analysis", "", report.root_cause_analysis, "",
              "## Risk Factors", "", report.risk_factors, "",
              "## Recommendations", "", report.recommendations, "",
              "## Conclusion", "", report.conclusion, ""]
    if report.sources:
        lines += ["## Sources", ""] + [f"- {s}" for s in report.sources] + [""]
    lines += ["---", "", "*Report generated by the Deep Search Agent (Chapter 5 Lab, customer-care edition)*"]
    return "\n".join(lines)

from IPython.display import Markdown, display
display(Markdown(format_report_markdown(report)))

## 12. Results

Expect the memo to separate **symptoms** (FL-340 CSAT 3.4, down 0.7; contact rate 36 per 100 orders;
FCR 48%) from **root causes** (relay part R-22 failures and back-order; technicians without the part;
agents not offering the replacement the policy allows), and to note the competitor's 7-day replacement guarantee.

**What to notice**
* **Parallel research waves** cut wall-clock time; the dependency graph keeps the logic correct.
* **Deterministic validation** catches impossible numbers before they reach an executive memo.
* **Replanning** turns validation gaps into a better second plan, automatically.

## Discussion
1. Which recommendations are about *product*, *process* and *policy*? Who owns each?
2. Why should the validator, not the LLM, decide whether the data is good enough?
3. What would it take to run this investigation every Monday for every product line?